# 00 · Landing — Download source files

Downloads monthly Jersey City Citibike trip archives (`.zip`) from the public S3
bucket and extracts the single CSV from each into the landing **Unity Catalog
Volume**, partitioned by `year-month`.

- Probes the known Citibike archive URL variants (naming has changed over the years).
- Idempotent per month via `overwrite`; skips months with no published archive.
- Landing a file here **triggers the medallion pipeline** (file-arrival trigger).

**Downstream:** `01_bronze`

In [ ]:
env_name = dbutils.widgets.get("env")

first_month = 1
last_month = 12
start_year = 2016
last_year = 2026  # 2026
catalog_name = f"citibike_ext_{env_name}" if env_name == "dev" else f"citibike_{env_name}"
schema = "00_landing"
volume = "source_data"
folder = "citibike_trips"

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import requests

# Add the `src/` directory (the notebook's parent) to the import path
sys.path.append(str(Path.cwd().parent.parent))
from shared.utils import download_and_extract


def resolve_source_url(year: int, month: int) -> str | None:
    """Return the first Citibike archive URL that exists for a given month.

    Citibike has published Jersey City ("JC-") archives under several naming
    conventions over the years, so we probe the known variants in order.
    """
    stem = f"JC-{year}{month:02}"
    candidates = [
        f"https://s3.amazonaws.com/tripdata/{stem}-citibike-tripdata.csv.zip",
        f"https://s3.amazonaws.com/tripdata/{stem}-citibike-tripdata.zip",
        f"https://s3.amazonaws.com/tripdata/{stem} citibike-tripdata.csv.zip",
    ]
    for url in candidates:
        if requests.head(url, allow_redirects=True, timeout=30).status_code == 200:
            return url
    return None


for year in range(start_year, last_year + 1):
    last_month = 12 if year < last_year else datetime.now().month - 1
    for month in range(first_month, last_month + 1):
        file_url = resolve_source_url(year, month)
        if file_url is None:
            print(f"No source archive found for {year}-{month:02} — skipping")
            continue
        download_and_extract(
            url=file_url,
            catalog=catalog_name,
            schema=schema,
            volume=volume,
            folder=folder,
            year=year,
            month=month,
            overwrite=True,
        )